In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install -q scikit-learn networkx sentence-transformers matplotlib
# utile per i modelli recenti
!pip install -U transformers huggingface_hub

In [3]:
!pip install -U sentence-transformers git+https://github.com/huggingface/transformers@v4.56.0-Embedding-Gemma-preview

  Cloning https://github.com/huggingface/transformers (to revision v4.56.0-Embedding-Gemma-preview) to /tmp/pip-req-build-70a61n28
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-70a61n28
  Running command git checkout -q 60b68e304cf4b6569b0660a13b558b929d4b0e77
  Resolved https://github.com/huggingface/transformers to commit 60b68e304cf4b6569b0660a13b558b929d4b0e77
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for transformers: filename=transformers-4.57.0.dev0-py3-none-any.whl size=12604658 sha256=1b79457428a70f86b82b227545e96b5e596a3a09bc81bdb86d8df596549a3129
  Stored in directory: /tmp/pip-ephem-wheel-cache-0moxt6lz/wheels/3a/21/76/c31899bac2cf601d3c74091b26a413bc3fb54770d5ccb5c924
Successfully built transformers
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.3
    Uni

In [4]:
# SETUP

!pip install -q scikit-learn networkx sentence-transformers matplotlib
!pip install -U transformers huggingface_hub
!pip install -U sentence-transformers git+https://github.com/huggingface/transformers@v4.56.0-Embedding-Gemma-preview

# IMPORTS
import json
import re
import time
import logging
import subprocess
import traceback
import numpy as np
import os
from pathlib import Path
from typing import Dict, Any, List, Optional, Tuple, Set
from datetime import datetime
from collections import Counter, defaultdict

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.cluster import AgglomerativeClustering
import networkx as nx
try:
    import matplotlib
    matplotlib.use('Agg')  # Backend non-interattivo
    import matplotlib.pyplot as plt
    PLOT_AVAILABLE = True
except ImportError:
    PLOT_AVAILABLE = False
    print("⚠ Matplotlib non disponibile")
import traceback
import matplotlib.pyplot as plt
from sentence_transformers import SentenceTransformer
from huggingface_hub import login

# LOGGING
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# La parte di mount drive viene gestita nella cella Colab (vedi sezione 2)
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        print("Tentativo di montaggio di Google Drive...")
        # Non forziamo il remount, ma lo chiamiamo per assicurare il mount
        drive.mount('/content/drive', force_remount=False)
    else:
        print("Drive già montato.")
except:
    print("Ambiente non Colab o errore nel montaggio Drive.")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 1.4 MB/s eta 0:00:00
  Using cached huggingface_hub-1.2.2-py3-none-any.whl.metadata (13 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 32.9 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 4.57.0.dev0
    Uninstalling transformers-4.57.0.dev0:
      Successfully uninstalled transformers-4.57.0.dev0
  Cloning https://github.com/huggingface/transformers (to revision v4.56.0-Embedding-Gemma-preview) to /tmp/pip-req-build-ahciylyo
  Running command git clone --filter=blob:none --quiet https://github.com/huggingface/transformers /tmp/pip-req-build-ahciylyo
  Running command git checkout -q 60b68e304cf4b6569b0660a13b558b929d4b0e77
  Resolved https://github.com/huggingface/transformers to commit 60b68e304cf4b6569b0660a13b558b929d4b0e77
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  

In [5]:
# CONFIGURAZIONE OSHA
class Config:
    # Path su Drive
    OUTPUT_DIR = Path('/content/drive/MyDrive/OSHA_Thesis_Data')
    PDF_DIR = OUTPUT_DIR / 'pdfs'
    JSON_DIR = OUTPUT_DIR / 'json'
    IMAGES_DIR = OUTPUT_DIR / 'images'
    TABLES_DIR = OUTPUT_DIR / 'tables'
    KEYWORDS_DIR = OUTPUT_DIR / 'keywords'  # Keyword pre-estratte
    ENRICHED_DIR = OUTPUT_DIR / 'enriched_json'  # Output arricchiti
    LOG_DIR = OUTPUT_DIR / 'logs'
    GRAPH_DIR = OUTPUT_DIR / 'graphs'

    # LLM: Llama 3.1 8B
    OLLAMA_MODEL = "llama3.1:8b"
    USE_LLM_SUMMARY = True

    # DATI DOMINIO
    # DOMINIO OSHA (OSH - Occupational Safety and Health)
    DOCUMENT_CATEGORIES = [
        "Musculoskeletal disorders",
        "Psychosocial risks and mental health",
        "Chemical and biological risks",
        "Physical risks (noise, vibration, radiation)",
        "Fire and explosion prevention",
        "Machinery and equipment safety",
        "Construction and building sites",
        "Agriculture and forestry",
        "Transport and logistics",
        "Healthcare sector",
        "Personal protective equipment (PPE)",
        "Training and education",
        "Health surveillance",
        "Risk assessment and management",
        "Accidents and occupational diseases",
        "Work organization and ergonomics",
        "Disability and inclusion at work",
        "Other"
    ]

    CATEGORY_TERMS = {
        "Musculoskeletal disorders": [
            "musculoskeletal", "msd", "back pain", "repetitive strain", "ergonomics",
            "manual handling", "lifting", "posture", "workstation", "repetitive movements"
        ],
        "Psychosocial risks and mental health": [
            "stress", "burnout", "mental health", "psychosocial", "harassment",
            "bullying", "work-life balance", "psychological", "wellbeing", "anxiety",
            "depression", "workload", "emotional"
        ],
        "Chemical and biological risks": [
            "chemical", "hazardous substances", "carcinogens", "biological agents",
            "exposure", "toxicity", "contamination", "asbestos", "solvents", "dust"
        ],
        "Physical risks (noise, vibration, radiation)": [
            "noise", "vibration", "radiation", "temperature", "lighting",
            "electromagnetic fields", "physical hazards"
        ],
        "Fire and explosion prevention": [
            "fire", "explosion", "flammable", "emergency", "evacuation", "fire safety"
        ],
        "Machinery and equipment safety": [
            "machinery", "equipment", "tools", "maintenance", "guards", "safety devices"
        ],
        "Construction and building sites": [
            "construction", "building site", "scaffolding", "excavation", "demolition",
            "falls from height"
        ],
        "Agriculture and forestry": [
            "agriculture", "farming", "forestry", "rural", "agricultural machinery",
            "pesticides", "livestock"
        ],
        "Transport and logistics": [
            "transport", "logistics", "driving", "vehicles", "warehouse", "forklift"
        ],
        "Healthcare sector": [
            "healthcare", "hospital", "nurses", "medical staff", "patients",
            "long-term care", "caregivers", "health workers"
        ],
        "Personal protective equipment (PPE)": [
            "ppe", "protective equipment", "respirators", "gloves", "helmets",
            "safety shoes", "hearing protection"
        ],
        "Training and education": [
            "training", "education", "awareness", "competence", "skills",
            "instruction", "learning"
        ],
        "Health surveillance": [
            "health surveillance", "medical examination", "monitoring",
            "occupational health", "screening"
        ],
        "Risk assessment and management": [
            "risk assessment", "risk management", "hazard identification",
            "prevention", "control measures", "safety management"
        ],
        "Accidents and occupational diseases": [
            "accident", "injury", "occupational disease", "illness", "fatality",
            "incident", "statistics"
        ],
        "Work organization and ergonomics": [
            "work organization", "ergonomics", "job design", "work schedule",
            "shift work", "workload", "task design"
        ],
        "Disability and inclusion at work": [
            "disability", "inclusion", "accessible", "accommodation",
            "reasonable adjustments", "diversity"
        ]
    }

    DOMAIN_TERMS = {
        'risk', 'hazard', 'safety', 'health', 'occupational', 'workplace', 'worker',
        'employee', 'employer', 'prevention', 'protection', 'assessment', 'management',
        'exposure', 'injury', 'illness', 'disease', 'accident', 'incident', 'fatality',
        'control', 'measure', 'procedure', 'training', 'equipment', 'ppe', 'surveillance',
        'monitoring', 'inspection', 'compliance', 'regulation', 'directive', 'legislation',
        'standard', 'guideline', 'policy', 'emergency', 'evacuation', 'ergonomics',
        'psychosocial', 'stress', 'burnout', 'mental', 'physical', 'chemical', 'biological',
        'musculoskeletal', 'repetitive', 'manual handling', 'machinery', 'construction',
        'agriculture', 'healthcare', 'transport', 'manufacturing', 'rehabilitation',
        'compensation', 'disability', 'inclusion', 'accessible', 'carcinogen', 'mutagen',
        'hazardous substance', 'dust', 'noise', 'vibration', 'radiation', 'fire',
        'explosion', 'confined space', 'working at height', 'scaffolding'
    }

    # Professioni OSH (EU context)
    KNOWN_PROFESSIONS = [
        "Construction workers", "Carpenters", "Masons", "Scaffolders", "Electricians",
        "Plumbers", "Welders", "Industrial mechanics", "Machine operators", "Maintenance workers",
        "Farmers", "Forestry workers", "Agricultural workers", "Nurses", "Doctors",
        "Healthcare assistants", "Laboratory technicians", "Radiologists", "Caregivers",
        "Long-term care workers", "Professional drivers", "Warehouse workers", "Forklift operators",
        "OSH professionals", "Safety managers", "OSH coordinators", "Safety representatives",
        "Chemists", "Biologists", "Chemical laboratory technicians", "Workers exposed to hazardous chemicals",
        "Workers exposed to carcinogens", "Workers exposed to biological agents",
        "Workers exposed to dust and mineral fibres", "Workers exposed to noise and vibration",
        "Workers in confined spaces", "Workers at height", "Firefighters",
        "General workers", "All workers", "Office workers", "Administrative staff",
        "Managers", "Supervisors", "Manufacturing workers", "Factory workers"
    ]

    PROFESSION_BLACKLIST = {
        "General workers", "All workers", "Generic staff", "Non-specific workers"
    }

    MIN_TEXT_LENGTH = 100

    # CONFIGURAZIONI EMBEDDING GEMMA
    GEMMA_MODEL_NAME = "google/embeddinggemma-300m"
    USE_EMBEDDING_GEMMA = True

    # PERCORSO DEFINITIVO SU DRIVE DOVE È STATO SALVATO IL MODELLO
    LOCAL_EMBEDDING_MODEL_PATH = Path('/content/drive/MyDrive/INAIL_Thesis_Data_old/EmbeddingGemma_Offline')

    # IMPOSTATO A TRUE PER FORZARE IL CARICAMENTO OFFLINE
    USE_LOCAL_HPC_MODEL = True

    # HuggingFace Token
    HARDCODED_HF_TOKEN = ""  # set HUGGINGFACE_TOKEN in the environment
    HF_TOKEN_ENV = "HUGGINGFACE_TOKEN"

    TARGET_KEYWORDS = 8  # Numero totale keyword desiderate

    # Threshold aumentati per maggiore precisione
    PROFESSION_THRESHOLD = 0.55
    CATEGORY_THRESHOLD = 0.30


    @classmethod
    def get_hf_token(cls) -> str:
        """Recupera token HuggingFace"""
        if cls.HARDCODED_HF_TOKEN:
            return cls.HARDCODED_HF_TOKEN
        return os.environ.get(cls.HF_TOKEN_ENV)

    @classmethod
    def login_huggingface(cls):
        """Login automatico su HuggingFace"""
        token = cls.get_hf_token()
        if token:
            try:
                login(token=token, add_to_git_credential=False)
                return True
            except Exception:
                return False
        return False

    @classmethod
    def setup(cls):
        """Verifica e crea le directory necessarie"""
        if not cls.OUTPUT_DIR.exists():
            logger.error("La cartella principale OSHA_Thesis_Data non esiste. Controlla il mount di Drive.")
            return False

        if not cls.JSON_DIR.exists():
            logger.error(f"Cartella JSON mancante in {cls.JSON_DIR}")
            return False

        if not cls.KEYWORDS_DIR.exists():
            logger.warning(f"⚠ Cartella KEYWORDS mancante in {cls.KEYWORDS_DIR}")
            logger.warning("Si procederà senza keyword pre-estratte OSHA")
            # ← NON ritornare False, continua lo stesso

        json_files = list(cls.JSON_DIR.glob("*.json"))
        keyword_files = list(cls.KEYWORDS_DIR.glob("*.json")) if cls.KEYWORDS_DIR.exists() else []

        if len(json_files) == 0:
            logger.error("Nessun file JSON trovato nella cartella json.")
            return False

        print(f"✓ Trovati {len(json_files)} file JSON")
        print(f"✓ Trovati {len(keyword_files)} file con keyword pre-estratte OSHA")

        # Crea directory output se mancanti
        for d in [cls.ENRICHED_DIR, cls.LOG_DIR, cls.GRAPH_DIR]:
            d.mkdir(exist_ok=True, parents=True)

        return True

In [6]:
# EMBEDDING MODEL LOADER

def load_embedding_model(use_gemma: bool = True) -> Optional[SentenceTransformer]:
    """Carica EmbeddingGemma (offline da Drive/HPC, con fallback online/cache)."""

    # CARICAMENTO OFFLINE/HPC (PRIORITARIO DA DRIVE)
    local_path = Config.LOCAL_EMBEDDING_MODEL_PATH
    if Config.USE_LOCAL_HPC_MODEL and local_path.exists():
        print(f"\n[Embedding Model] Tentativo caricamento OFFLINE forzato da: {local_path}")
        try:
            model = SentenceTransformer(str(local_path))
            print(f"✓ Modello EmbeddingGemma locale caricato con successo (HPC ready)")
            print(f"  Device: {model.device}")
            return model
        except Exception as e:
            logger.error(f"✗ Errore caricamento modello locale da Drive: {e}")
            print("  Tentativo di procedere con l'opzione online/cache...")

    # LOGICA ONLINE/CACHE (FALLBACK)
    if use_gemma:
        model_name = Config.GEMMA_MODEL_NAME
        print(f"\n[Embedding Model] Tentativo caricamento {model_name} ONLINE/Cache...")

        if not Config.login_huggingface():
            logger.warning(f"⚠ Login fallito o token non valido. Fallback su BERT.")
            use_gemma = False
        else:
            try:
                model = SentenceTransformer(model_name)
                print(f"✓ Modello {model_name} caricato con successo da Internet/Cache")
                print(f"  Device: {model.device}")
                return model
            except Exception as e:
                logger.error(f"✗ Errore caricamento {model_name}: {e}")
                use_gemma = False

    # FALLBACK: BERT Multilingue (funziona anche per inglese)
    fallback_model = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
    print(f"\n[Embedding Model] Fallback al modello: {fallback_model}")
    model = SentenceTransformer(fallback_model)
    print(f"Fallback model caricato. Device: {model.device}")
    return model

In [7]:
# LAW EXTRACTOR

class EULawExtractor:
    """Estrattore di riferimenti normativi EU-OSH (direttive, regolamenti, framework)"""

    PATTERNS = {
        # DIRETTIVE EU
        'eu_directive': [
            r'Directive\s+(\d{4})/(\d+)/(?:EC|EU)',
            r'Council\s+Directive\s+(\d{2,4})/(\d+)',
            r'Directive\s+No\.?\s*(\d{4})/(\d+)',
        ],

        # REGOLAMENTI EU
        'eu_regulation': [
            r'Regulation\s+\(EU\)\s+(?:No\.?\s*)?(\d{4})/(\d+)',
            r'Regulation\s+\(EC\)\s+(?:No\.?\s*)?(\d{1,4})/(\d{2,4})',
            r'Council\s+Regulation\s+\(EU\)\s+(\d{4})/(\d+)',
        ],

        # FRAMEWORK DIRECTIVE (89/391/EEC e derivate)
        'framework': [
            r'Framework\s+Directive\s+(\d{2})/(\d{3})',
            r'Directive\s+(\d{2})/(\d{3})/EEC',
        ],

        # ARTICOLI
        'article': [
            r'Article\s+(\d+(?:[a-z])?)',
            r'Art\.\s+(\d+)',
            r'Annex\s+([IVX]+|\w+)',
        ],

        # STANDARD INTERNAZIONALI (ISO, EN, CEN)
        'standard': [
            r'(?:ISO|EN|CEN)\s+(\d+(?:[-:]\d+)?)',
            r'EN\s+ISO\s+(\d+)',
        ],

        # GUIDELINES & CODES
        'guideline': [
            r'EU-OSHA\s+(?:Guideline|Guide)\s+(\d+)',
            r'Code\s+of\s+Practice\s+(?:on\s+)?(.{10,50})',
        ],
    }

    NORMALIZATIONS = {
        'eu_directive': lambda m: f"Directive {m.group(1)}/{m.group(2)}/EU",
        'eu_regulation': lambda m: f"Regulation (EU) {m.group(1)}/{m.group(2)}",
        'framework': lambda m: f"Framework Directive {m.group(1)}/{m.group(2)}/EEC",
        'article': lambda m: f"Article {m.group(1)}",
        'standard': lambda m: f"Standard {m.group(1)}",
        'guideline': lambda m: f"Guideline {m.group(1)}",
    }

    PRIORITY_ORDER = ['framework', 'eu_directive', 'eu_regulation', 'standard', 'article', 'guideline']

    def __init__(self):
        self.compiled_patterns = {}
        for tipo, patterns in self.PATTERNS.items():
            self.compiled_patterns[tipo] = [re.compile(p, re.IGNORECASE) for p in patterns]

    @staticmethod
    def _normalize_year(year_str: str) -> str:

        # Normalizza l'anno (es. 08 -> 2008, 96 -> 1996)
        if not year_str:
             return ""
        year = int(year_str)
        if year >= 1000:
            return str(year)
        elif year < 50:
            return str(2000 + year)
        else:
            return str(1900 + year)

    def __init__(self):
        self.compiled_patterns = {}
        for tipo, patterns in self.PATTERNS.items():
            self.compiled_patterns[tipo] = [re.compile(p, re.IGNORECASE) for p in patterns]

    def _extract_sort_key(self, law_ref: str) -> tuple:
        """Estrae chiave per ordinamento (numero, anno)"""
        numbers = re.findall(r'\d+', law_ref)
        if len(numbers) >= 2:
            return (int(numbers[0]), int(numbers[1]))
        elif len(numbers) == 1:
            return (int(numbers[0]), 0)
        return (0, 0)

    def _deduplicate_preserving_order(self, items: List[str]) -> List[str]:
        """Rimuove duplicati mantenendo ordine"""
        seen = set()
        result = []
        for item in items:
            if item not in seen:
                seen.add(item)
                result.append(item)
        return result

    def extract_with_confidence(self, text: str, include_articles: bool = False) -> Tuple[List[str], Dict[str, float]]:
        """Estrae riferimenti normativi EU con confidence score"""
        found_laws_by_type = defaultdict(list)
        law_counts = Counter()

        for law_type, compiled_patterns in self.compiled_patterns.items():
            if law_type == 'article' and not include_articles:
                continue

            for pattern in compiled_patterns:
                for match in pattern.finditer(text):
                    try:
                        normalizer = self.NORMALIZATIONS.get(law_type)
                        if normalizer:
                            normalized = normalizer(match)
                            found_laws_by_type[law_type].append(normalized)
                            law_counts[normalized] += 1
                    except Exception:
                        continue

        # Ordina per priorità
        result_ordered = []
        for law_type in self.PRIORITY_ORDER:
            if law_type in found_laws_by_type:
                sorted_laws = sorted(found_laws_by_type[law_type], key=self._extract_sort_key)
                result_ordered.extend(sorted_laws)

        unique_laws = self._deduplicate_preserving_order(result_ordered)
        confidence = self._calculate_confidence(unique_laws, law_counts)

        return unique_laws[:20], confidence

    def _calculate_confidence(self, laws: List[str], counts: Counter) -> Dict[str, float]:
        """Calcola confidence score per le leggi estratte"""
        if not laws:
            return {
                'overall': 0.70,
                'count': 0.5,
                'coverage': 0.7,
                'coherence': 0.9
            }

        count_score = min(1.0, len(laws) / 5.0)
        avg_mentions = np.mean([counts[law] for law in laws])
        coverage_score = min(1.0, avg_mentions / 3.0)

        # Coerenza basata su anni (per direttive EU)
        years = []
        for law in laws:
            match = re.search(r'/(\d{4})/', law)
            if match:
                years.append(int(match.group(1)))

        if len(years) > 1:
            year_std = np.std(years)
            coherence_score = max(0.0, 1.0 - year_std / 30.0)
        else:
            coherence_score = 0.7

        overall = 0.4 * count_score + 0.35 * coverage_score + 0.25 * coherence_score

        return {
            'overall': round(overall, 3),
            'count': round(count_score, 3),
            'coverage': round(coverage_score, 3),
            'coherence': round(coherence_score, 3)
        }

In [8]:
# CARICAMENTO KEYWORD PRE-ESTRATTE E COMPLETAMENTO

def load_preextracted_keywords(doc_url: str, keywords_dir: Path) -> List[str]:
    """
    Carica le keyword pre-estratte per un documento OSHA dato il suo URL.

    Args:
        doc_url: URL del documento OSHA
        keywords_dir: Path alla directory con i file JSON delle keyword

    Returns:
        Lista di keyword pre-estratte (vuota se non trovate)
    """
    try:
        # Cerca tutti i file JSON nella directory keywords
        for kw_file in keywords_dir.glob("*.json"):
            with open(kw_file, 'r', encoding='utf-8') as f:
                kw_data = json.load(f)

                # Match per URL
                if kw_data.get('url') == doc_url:
                    keywords = kw_data.get('keywords', [])
                    print(f"[PreExtracted Keywords] Trovate {len(keywords)} keyword per documento")
                    for i, kw in enumerate(keywords, 1):
                        print(f"  {i}. {kw}")
                    return keywords

        print(f"[PreExtracted Keywords] Nessuna keyword pre-estratta trovata per URL: {doc_url}")
        return []

    except Exception as e:
        logger.error(f"Errore caricamento keyword pre-estratte: {e}")
        return []


def complete_keywords_to_target(
    preextracted_keywords: List[str],
    document_text: str,
    embedding_model: SentenceTransformer,
    domain_terms: Set[str],
    target_count: int = 8
) -> Tuple[List[str], Dict[str, Any]]:
    """
    Completa le keyword pre-estratte fino al numero target usando TF-IDF + EmbedRank.

    Args:
        preextracted_keywords: Keyword già estratte da OSHA
        document_text: Testo completo del documento
        embedding_model: Modello per embeddings
        domain_terms: Termini di dominio OSH
        target_count: Numero totale keyword desiderate (default 8)

    Returns:
        Tuple (lista keyword finali, metadata con confidence)
    """
    print(f"\n[Keyword Completion] Target: {target_count} keyword")
    print(f"[Keyword Completion] Pre-estratte: {len(preextracted_keywords)}")

    # Caso 1: Abbiamo già abbastanza keyword
    if len(preextracted_keywords) >= target_count:
        print(f"[Keyword Completion] Selezione prime {target_count} keyword pre-estratte")
        final_keywords = preextracted_keywords[:target_count]

        # Calcola confidence sulle keyword selezionate
        confidence = _calculate_keyword_confidence_simple(final_keywords, document_text, domain_terms)

        metadata = {
            'keywords': final_keywords,
            'preextracted_count': len(preextracted_keywords),
            'added_count': 0,
            'confidence': confidence,
            'method': 'Pre-extracted only (sufficient)'
        }

        return final_keywords, metadata

    # Caso 2: Dobbiamo aggiungere keyword
    needed_count = target_count - len(preextracted_keywords)
    print(f"[Keyword Completion] Estrazione {needed_count} keyword aggiuntive...")

    # Estrai candidate keywords con TF-IDF Domain-Aware
    tfidf_keywords = _extract_tfidf_domain_aware(
        document_text,
        domain_terms,
        top_n=needed_count * 3  # Estrai più candidate per diversità
    )

    if not tfidf_keywords:
        print("[Keyword Completion] ⚠ Nessuna keyword aggiuntiva estratta")
        final_keywords = preextracted_keywords
        confidence = _calculate_keyword_confidence_simple(final_keywords, document_text, domain_terms)

        metadata = {
            'keywords': final_keywords,
            'preextracted_count': len(preextracted_keywords),
            'added_count': 0,
            'confidence': confidence,
            'method': 'Pre-extracted only (extraction failed)'
        }

        return final_keywords, metadata

    # Filtra keyword che non sono duplicate delle pre-estratte
    preextracted_lower = [kw.lower() for kw in preextracted_keywords]
    candidate_keywords = [
        kw for kw, score in tfidf_keywords
        if kw.lower() not in preextracted_lower
    ]

    # Seleziona keyword diverse usando embeddings
    added_keywords = _select_diverse_keywords(
        candidate_keywords[:needed_count * 2],  # Più candidate per diversity
        embedding_model,
        needed_count,
        existing_keywords=preextracted_keywords
    )

    # Combina: pre-estratte hanno priorità assoluta
    final_keywords = preextracted_keywords + added_keywords

    print(f"[Keyword Completion] ✓ Aggiunte {len(added_keywords)} keyword")
    print(f"[Keyword Completion] Totale finale: {len(final_keywords)} keyword")

    # Calcola confidence
    confidence = _calculate_keyword_confidence_combined(
        final_keywords,
        preextracted_keywords,
        added_keywords,
        document_text,
        domain_terms
    )

    metadata = {
        'keywords': final_keywords,
        'preextracted_count': len(preextracted_keywords),
        'added_count': len(added_keywords),
        'confidence': confidence,
        'method': 'Hybrid (pre-extracted + TF-IDF-DomainAware + EmbedRank)'
    }

    return final_keywords, metadata


def _extract_tfidf_domain_aware(text: str, domain_terms: Set[str], top_n: int = 30) -> List[Tuple[str, float]]:
    """Estrae keyword candidate usando TF-IDF con boost per termini di dominio"""

    # Preprocessing
    text_clean = _preprocess_english_text(text)

    if len(text_clean) < 50:
        return []

    try:
        # Stopwords inglesi
        english_stopwords = _load_english_stopwords()

        vectorizer = TfidfVectorizer(
            max_features=100,
            ngram_range=(1, 3),
            stop_words=list(english_stopwords),
            min_df=1,
            lowercase=True,
            token_pattern=r'(?u)\b[a-z]{3,}\b'  # Min 3 caratteri per inglese
        )

        tfidf_matrix = vectorizer.fit_transform([text_clean])
        feature_names = vectorizer.get_feature_names_out()
        scores = tfidf_matrix.toarray()[0]

        # Boost per termini di dominio
        boosted_scores = []
        for i, term in enumerate(feature_names):
            score = scores[i]

            # Boost se il termine è nel domain o contiene un domain term
            if any(domain_term in term for domain_term in domain_terms):
                score *= 1.5

            boosted_scores.append((term, score))

        boosted_scores.sort(key=lambda x: x[1], reverse=True)

        return boosted_scores[:top_n]

    except Exception as e:
        logger.error(f"Errore TF-IDF extraction: {e}")
        return []


def _select_diverse_keywords(
    candidates: List[str],
    embedding_model: SentenceTransformer,
    needed_count: int,
    existing_keywords: List[str],
    similarity_threshold: float = 0.70
) -> List[str]:
    """Seleziona keyword diverse evitando sovrapposizione con quelle esistenti"""

    if not candidates:
        return []

    selected = []

    # Embeddings delle keyword esistenti
    if existing_keywords:
        existing_embeddings = embedding_model.encode(existing_keywords)
    else:
        existing_embeddings = None

    for candidate in candidates:
        if len(selected) >= needed_count:
            break

        candidate_emb = embedding_model.encode([candidate])[0]

        # Check similarità con keyword esistenti
        if existing_embeddings is not None:
            max_sim_existing = max(
                float(cosine_similarity([candidate_emb], [emb])[0][0])
                for emb in existing_embeddings
            )
            if max_sim_existing > similarity_threshold:
                continue  # Troppo simile alle pre-estratte

        # Check similarità con keyword già selezionate
        if selected:
            selected_embeddings = embedding_model.encode(selected)
            max_sim_selected = max(
                float(cosine_similarity([candidate_emb], [emb])[0][0])
                for emb in selected_embeddings
            )
            if max_sim_selected > similarity_threshold:
                continue  # Troppo simile alle già selezionate

        selected.append(candidate)

    return selected


def _preprocess_english_text(text: str) -> str:
    """Preprocessing per testo inglese"""
    text = text.lower()
    text = re.sub(r'[^\w\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text)
    return text.strip()

def _load_english_stopwords() -> Set[str]:

    return {
        'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
        'of', 'with', 'by', 'from', 'as', 'is', 'was', 'are', 'were', 'be',
        'been', 'being', 'have', 'has', 'had', 'do', 'does', 'did', 'will',
        'would', 'could','should', 'may', 'might', 'must', 'can', 'this', 'that', 'these', 'those',
        'it', 'its', 'which', 'who', 'what', 'where', 'when', 'how', 'why',
        'all', 'each', 'every', 'both', 'few', 'more', 'most', 'other', 'some',
        'such', 'no', 'nor', 'not', 'only', 'own', 'same', 'so', 'than', 'too',
        'very', 'can', 'just', 'should', 'now', 'document', 'documents', 'paper',
        'study', 'report', 'article', 'publication', 'section', 'chapter',
        'page', 'figure', 'table', 'appendix', 'annex', 'following', 'described',
        'used', 'using', 'use', 'uses', 'based', 'case', 'cases', 'example',
        'examples', 'also', 'however', 'therefore', 'thus', 'furthermore',
        'moreover', 'additionally', 'regarding', 'concerning', 'related',
        'particular', 'specific', 'general', 'overall', 'various', 'different',
        'according', 'including', 'provide', 'provides', 'provided',
        'shall', 'must', 'should', 'require', 'required', 'requires',
        'provide', 'provided', 'provides', 'ensure', 'ensuring',
        'include', 'includes', 'included', 'following', 'described',
        'establish', 'established', 'implement', 'implemented',
        'identify', 'identified', 'assess', 'assessed', 'determine',
        'comply', 'accordance', 'appropriate', 'necessary',
    }


def _calculate_keyword_confidence_simple(
    keywords: List[str],
    document_text: str,
    domain_terms: Set[str]
) -> Dict[str, float]:
    """Calcola confidence per keyword solo pre-estratte"""

    if not keywords:
        return {
            'overall': 0.0,
            'document_coverage': 0.0,
            'term_specificity': 0.0,
            'domain_relevance': 0.0
        }

    doc_lower = document_text.lower()

    # Coverage: quante keyword appaiono nel documento
    found_count = sum(1 for kw in keywords if kw.lower() in doc_lower)
    coverage_score = found_count / len(keywords)

    # Specificity: lunghezza media keyword
    avg_length = np.mean([len(kw) for kw in keywords])
    length_score = min(1.0, avg_length / 12.0)

    # Domain relevance: quante keyword contengono termini di dominio
    domain_count = sum(
        1 for kw in keywords
        if any(dt in kw.lower() for dt in domain_terms)
    )
    domain_score = domain_count / len(keywords)

    specificity_score = 0.5 * length_score + 0.5 * domain_score

    overall = 0.40 * coverage_score + 0.60 * specificity_score

    return {
        'overall': round(float(overall), 3),
        'document_coverage': round(float(coverage_score), 3),
        'term_specificity': round(float(specificity_score), 3),
        'domain_relevance': round(float(domain_score), 3)
    }


def _calculate_keyword_confidence_combined(
    final_keywords: List[str],
    preextracted_keywords: List[str],
    added_keywords: List[str],
    document_text: str,
    domain_terms: Set[str]
) -> Dict[str, float]:
    """Calcola confidence per keyword combinate (pre-estratte + aggiunte)"""

    if not final_keywords:
        return {
            'overall': 0.0,
            'document_coverage': 0.0,
            'term_specificity': 0.0,
            'domain_relevance': 0.0,
            'preextracted_ratio': 0.0,
            'added_quality': 0.0
        }

    doc_lower = document_text.lower()

    # Coverage generale
    found_count = sum(1 for kw in final_keywords if kw.lower() in doc_lower)
    coverage_score = found_count / len(final_keywords)

    # Specificity
    avg_length = np.mean([len(kw) for kw in final_keywords])
    length_score = min(1.0, avg_length / 12.0)

    domain_count = sum(
        1 for kw in final_keywords
        if any(dt in kw.lower() for dt in domain_terms)
    )
    domain_score = domain_count / len(final_keywords)

    specificity_score = 0.5 * length_score + 0.5 * domain_score

    # Ratio pre-estratte (più alto = meglio, sono affidabili)
    preextracted_ratio = len(preextracted_keywords) / len(final_keywords)

    # Qualità keyword aggiunte (coverage nel documento)
    if added_keywords:
        added_found = sum(1 for kw in added_keywords if kw.lower() in doc_lower)
        added_quality = added_found / len(added_keywords)
    else:
        added_quality = 1.0  # Nessuna aggiunta = perfetto

    # Overall: peso maggiore a pre-estratte (sono certificate OSHA)
    overall = (
        0.25 * coverage_score +
        0.25 * specificity_score +
        0.30 * preextracted_ratio +  # Peso alto: preferenza per pre-estratte
        0.20 * added_quality
    )

    return {
        'overall': round(float(overall), 3),
        'document_coverage': round(float(coverage_score), 3),
        'term_specificity': round(float(specificity_score), 3),
        'domain_relevance': round(float(domain_score), 3),
        'preextracted_ratio': round(float(preextracted_ratio), 3),
        'added_quality': round(float(added_quality), 3)
    }

In [9]:
def extract_osha_document_text(document_data: Dict[str, Any]) -> str:
    """
    Estrae il testo completo da un documento JSON OSHA con gestione robusta.
    """
    chunks = []

    # Abstract
    try:
        abstract = document_data.get('web_metadata', {}).get('abstract', '')
        if abstract and len(abstract.strip()) > 20:
            chunks.append(f"ABSTRACT:\n{abstract}\n")
    except:
        pass

    # Title
    try:
        title = document_data.get('web_metadata', {}).get('title', '')
        if title:
            chunks.append(f"TITLE: {title}\n")
    except:
        pass

    # Content
    try:
        content = document_data.get('document_content')

        # CONTROLLO: Se content è None o non è un dict, ritorna stringa vuota
        if content is None or not isinstance(content, dict):
            return ""

        # Markdown content (priorità per OSHA)
        if content.get('markdown_content'):
            chunks.append(content['markdown_content'])
        # Plain text fallback
        elif content.get('plain_text'):
            chunks.append(content['plain_text'])
    except Exception as e:
        pass

    # Unisci tutto
    full_text = '\n\n'.join(chunks)

    return full_text if full_text.strip() else ""

def get_document_url(document_data: Dict[str, Any]) -> str:
    """
    Estrae l'URL dal documento JSON OSHA.

    Args:
        document_data: Dizionario caricato dal JSON OSHA

    Returns:
        URL del documento (stringa vuota se non trovato)
    """
    # L'URL può essere in scraping_metadata o web_metadata
    url = document_data.get('scraping_metadata', {}).get('url', '')

    if not url:
        url = document_data.get('web_metadata', {}).get('url', '')

    return url

In [10]:
# DOMAIN-AWARE TF-IDF + EMBEDRANK

class DomainAwareTFIDFExtractor:
    """TF-IDF + EmbedRank extractor adattato per documenti OSH in inglese"""

    def __init__(self, embedding_model: SentenceTransformer, domain_terms: Set[str]):
        print("[TF-IDF+EmbedRank] Inizializzazione per OSH (English)...")
        self.embedding_model = embedding_model
        self.domain_terms = domain_terms
        self.stopwords = self._load_stopwords()

        model_name = getattr(embedding_model, 'model_card_data', {}).get('model_id', 'unknown')
        print(f"[TF-IDF+EmbedRank] Modello attivo: {model_name}")
        print("[TF-IDF+EmbedRank] ✓ Pronto per elaborazione inglese")

    def _load_stopwords(self) -> Set[str]:
        """Carica stopwords inglesi per OSH"""
        return {
            'the', 'a', 'an', 'and', 'or', 'but', 'in', 'on', 'at', 'to', 'for',
            'of', 'with', 'by', 'from', 'as', 'is', 'was', 'are', 'were', 'be',
            'been', 'being', 'have', 'has', 'had', 'do', 'does', 'did', 'will',
            'would', 'could', 'should', 'may', 'might', 'must', 'can', 'this',
            'that', 'these', 'those', 'it', 'its', 'which', 'who', 'what', 'where',
            'when', 'how', 'why', 'all', 'each', 'every', 'both', 'few', 'more',
            'most', 'other', 'some', 'such', 'no', 'nor', 'not', 'only', 'own',
            'same', 'so', 'than', 'too', 'very', 'just', 'now', 'document',
            'documents', 'paper', 'study', 'report', 'article', 'publication',
            'section', 'chapter', 'page', 'figure', 'table', 'following',
            'described', 'used', 'using', 'use', 'uses', 'based', 'case', 'cases',
            'example', 'examples', 'also', 'however', 'therefore', 'thus',
            'furthermore', 'moreover', 'additionally', 'regarding', 'concerning',
            'related', 'particular', 'specific', 'general', 'overall', 'various',
            'different', 'according', 'including', 'provide', 'provides', 'provided',
            'can', 'may', 'must', 'should', 'need', 'needs', 'required', 'requirements',
            'through', 'during', 'within', 'between', 'among', 'under', 'over'
        }

    def _preprocess(self, text: str) -> str:
        """Preprocessing per testo inglese"""
        text = text.lower()
        text = re.sub(r'[^\w\s]', ' ', text)
        text = re.sub(r'\s+', ' ', text)
        return text.strip()

    def extract_keywords_with_confidence(
        self,
        document_text: str,
        top_n: int = 8,
        save_graph: bool = True,
        graph_path: Optional[Path] = None,
        doc_title: str = "Document"
    ) -> Tuple[List[str], Dict[str, Any], Optional[nx.Graph]]:
        """
        Estrae keyword usando TF-IDF Domain-Aware + EmbedRank.
        NOTA: Questa funzione è usata SOLO quando non ci sono keyword pre-estratte sufficienti.
        """

        # Estrai candidate con TF-IDF
        tfidf_keywords = self._extract_tfidf_domain_aware(document_text)

        if not tfidf_keywords:
            return [], {'confidence': {'overall': 0.0}}, None

        # Applica EmbedRank per ranking semantico
        embedrank_keywords, G = self._embed_rank(tfidf_keywords, document_text)

        # Combina e diversifica
        final_keywords = self._combine_and_diversify(
            tfidf_keywords, embedrank_keywords, top_n
        )

        # Calcola confidence
        confidence = self._calculate_confidence(
            final_keywords, tfidf_keywords, embedrank_keywords, document_text, G
        )

        # Salva grafo se richiesto
        if save_graph and G and graph_path:
            embedrank_scores = dict(embedrank_keywords)
            self._save_graph(G, embedrank_scores, graph_path, doc_title)

        metadata = {
            'keywords': final_keywords,
            'tfidf_keywords': [kw for kw, _ in tfidf_keywords[:10]],
            'embedrank_keywords': [kw for kw, _ in embedrank_keywords[:10]],
            'confidence': confidence,
            'method': 'TF-IDF-DomainAware + EmbedRank (OSH English)'
        }

        return final_keywords, metadata, G

    def _extract_tfidf_domain_aware(self, text: str, top_n: int = 30) -> List[Tuple[str, float]]:
        """Estrae keyword candidate con TF-IDF e domain boosting"""
        text_clean = self._preprocess(text)

        if len(text_clean) < 50:
            return []

        try:
            vectorizer = TfidfVectorizer(
                max_features=100,
                ngram_range=(1, 3),
                stop_words=list(self.stopwords),
                min_df=1,
                lowercase=True,
                token_pattern=r'(?u)\b[a-z]{3,}\b'  # Min 3 char per inglese
            )

            tfidf_matrix = vectorizer.fit_transform([text_clean])
            feature_names = vectorizer.get_feature_names_out()
            scores = tfidf_matrix.toarray()[0]

            # Boost per domain terms
            boosted_scores = []
            for i, term in enumerate(feature_names):
                score = scores[i]

                # Boost se contiene domain term
                if any(domain_term in term for domain_term in self.domain_terms):
                    score *= 1.5

                boosted_scores.append((term, score))

            boosted_scores.sort(key=lambda x: x[1], reverse=True)

            return boosted_scores[:top_n]

        except Exception as e:
            logger.error(f"Errore TF-IDF extraction: {e}")
            return []

    def _embed_rank(self, candidates: List[Tuple[str, float]], document_text: str) -> Tuple[List[Tuple[str, float]], nx.Graph]:
        """Applica EmbedRank (PageRank su grafo di similarità semantica)"""

        if len(candidates) < 3:
            return candidates, nx.Graph()

        candidate_terms = [term for term, _ in candidates[:30]]

        # Genera embeddings
        doc_emb = self.embedding_model.encode(
            [document_text[:3000]],
            convert_to_numpy=True,
            show_progress_bar=False,
        )[0]

        term_embs = self.embedding_model.encode(
            candidate_terms,
            convert_to_numpy=True,
            show_progress_bar=False,
        )

        # Costruisci grafo
        G = nx.Graph()

        for i, term_i in enumerate(candidate_terms):
            doc_sim = float(cosine_similarity([term_embs[i]], [doc_emb])[0][0])
            G.add_node(term_i, doc_similarity=doc_sim)

            for j in range(i + 1, len(candidate_terms)):
                term_sim = float(cosine_similarity([term_embs[i]], [term_embs[j]])[0][0])

                if term_sim > 0.3:
                    G.add_edge(candidate_terms[i], candidate_terms[j], weight=term_sim)

        if len(G.nodes()) == 0:
            return candidates, G

        try:
            # PageRank
            pagerank_scores = nx.pagerank(G, alpha=0.85, max_iter=100, weight='weight')

            final_scores = []
            for term in candidate_terms:
                pr = pagerank_scores.get(term, 0)
                doc_sim = G.nodes[term].get('doc_similarity', 0)

                # Combina PageRank + Document Similarity
                combined = 0.6 * pr + 0.4 * doc_sim
                final_scores.append((term, combined))

            final_scores.sort(key=lambda x: x[1], reverse=True)

            return final_scores, G

        except Exception as e:
            logger.error(f"Errore EmbedRank: {e}")
            return candidates, G

    def _combine_and_diversify(
        self,
        tfidf_kw: List[Tuple[str, float]],
        embedrank_kw: List[Tuple[str, float]],
        top_n: int
    ) -> List[str]:
        """Combina TF-IDF e EmbedRank con diversificazione"""

        tfidf_dict = dict(tfidf_kw)
        embedrank_dict = dict(embedrank_kw)

        all_terms = set(tfidf_dict.keys()) | set(embedrank_dict.keys())

        # Normalizza e combina score
        combined_scores = {}
        for term in all_terms:
            tfidf_score = tfidf_dict.get(term, 0)
            embedrank_score = embedrank_dict.get(term, 0)

            max_tfidf = max(tfidf_dict.values()) if tfidf_dict else 1.0
            max_embedrank = max(embedrank_dict.values()) if embedrank_dict else 1.0

            tfidf_norm = tfidf_score / max_tfidf
            embedrank_norm = embedrank_score / max_embedrank

            combined_scores[term] = 0.5 * tfidf_norm + 0.5 * embedrank_norm

        sorted_terms = sorted(combined_scores.items(), key=lambda x: x[1], reverse=True)

        # Diversificazione: evita keyword troppo simili
        selected = []
        term_embeddings = {}

        for term, score in sorted_terms:
            if len(selected) >= top_n:
                break

            if len(selected) == 0:
                selected.append(term)
                term_embeddings[term] = self.embedding_model.encode([term])[0]
                continue

            term_emb = self.embedding_model.encode([term])[0]

            # Check similarità con keyword già selezionate
            max_sim = 0
            for sel_term in selected:
                sim = float(cosine_similarity([term_emb], [term_embeddings[sel_term]])[0][0])
                max_sim = max(max_sim, sim)

            # Aggiungi solo se sufficientemente diversa
            if max_sim < 0.70:
                selected.append(term)
                term_embeddings[term] = term_emb

        return selected

    def _calculate_confidence(
        self,
        final_keywords: List[str],
        tfidf_kw: List[Tuple[str, float]],
        embedrank_kw: List[Tuple[str, float]],
        document_text: str,
        graph: nx.Graph
    ) -> Dict[str, float]:
        """Calcola confidence score per le keyword estratte"""

        if not final_keywords:
            return {
                'overall': 0.0,
                'score_distribution': 0.0,
                'document_coverage': 0.0,
                'term_specificity': 0.0,
                'graph_coherence': 0.0
            }

        tfidf_dict = dict(tfidf_kw)
        selected_scores = [tfidf_dict.get(kw, 0) for kw in final_keywords]

        # Score distribution
        if len(selected_scores) >= 2:
            score_gap = (selected_scores[0] - selected_scores[-1]) / (selected_scores[0] + 1e-6)
            distribution_score = min(1.0, score_gap * 2)
        else:
            distribution_score = 0.5

        # Document coverage
        doc_lower = document_text.lower()
        found_count = sum(1 for kw in final_keywords if kw in doc_lower)
        coverage_score = found_count / len(final_keywords)

        # Term specificity
        avg_length = np.mean([len(kw) for kw in final_keywords])
        length_score = min(1.0, avg_length / 12.0)

        domain_count = sum(1 for kw in final_keywords
                          if any(dt in kw for dt in self.domain_terms))
        domain_score = domain_count / len(final_keywords)

        specificity_score = 0.5 * length_score + 0.5 * domain_score

        # Graph coherence
        if graph and len(graph.nodes()) > 1:
            density = nx.density(graph)
            coherence_score = min(1.0, density * 5)
        else:
            coherence_score = 0.5

        # Overall
        overall = (
            0.35 * distribution_score +
            0.25 * coverage_score +
            0.20 * specificity_score +
            0.20 * coherence_score
        )

        return {
            'overall': round(float(overall), 3),
            'score_distribution': round(float(distribution_score), 3),
            'document_coverage': round(float(coverage_score), 3),
            'term_specificity': round(float(specificity_score), 3),
            'graph_coherence': round(float(coherence_score), 3)
        }

    def _save_graph(self, G: nx.Graph, scores: Dict[str, float],
                    output_path: Path, doc_title: str):
        """Salva grafo EmbedRank ottimizzato e leggibile per OSHA"""
        if not PLOT_AVAILABLE or len(G.nodes()) == 0:
            logger.warning("[Graph] Skipped: no nodes or matplotlib unavailable")
            return

        try:
            # Top 15 nodi più rilevanti (migliore leggibilità)
            top_nodes = sorted(scores.items(), key=lambda x: x[1], reverse=True)[:15]
            top_node_names = [node for node, _ in top_nodes]
            G_sub = G.subgraph(top_node_names)

            if len(G_sub.nodes()) < 2:
                logger.warning("[Graph] Too few nodes for visualization")
                return

            # Layout migliorato
            pos = nx.spring_layout(G_sub, k=1.5, iterations=100, seed=42)

            # Figura più grande
            plt.figure(figsize=(16, 12))

            # Calcola dimensioni e colori
            max_score = max(scores.values()) if scores else 1.0
            node_sizes = [min(scores.get(node, 0) / max_score * 8000, 8000)
                         for node in G_sub.nodes()]
            node_colors = [scores.get(node, 0) for node in G_sub.nodes()]

            # Disegna nodi
            nx.draw_networkx_nodes(
                G_sub, pos,
                node_size=node_sizes,
                node_color=node_colors,
                cmap=plt.cm.YlOrRd,
                alpha=0.85,
                edgecolors='darkblue',
                linewidths=2.0
            )

            # Disegna solo edges forti (similarity > 0.5)
            strong_edges = [(u, v) for u, v, d in G_sub.edges(data=True)
                           if d.get('weight', 0) > 0.5]
            nx.draw_networkx_edges(
                G_sub, pos,
                edgelist=strong_edges,
                alpha=0.3,
                width=1.5,
                edge_color='gray'
            )

            # Labels con sfondo bianco
            for node, (x, y) in pos.items():
                plt.text(
                    x, y, node,
                    fontsize=10,
                    fontweight='bold',
                    ha='center',
                    va='center',
                    bbox=dict(boxstyle='round,pad=0.3', facecolor='white',
                             edgecolor='darkblue', alpha=0.8)
                )

            # Titolo
            plt.title(f"Keyword Similarity Network: {doc_title[:50]}",
                     fontsize=14, fontweight='bold', pad=20)
            plt.axis('off')
            plt.tight_layout()

            # Salva
            safe_title = re.sub(r'[^\w\-_]', '_', doc_title[:40] or 'unknown')
            graph_file = output_path.parent / f"{safe_title}.png"
            graph_file.parent.mkdir(parents=True, exist_ok=True)

            plt.savefig(graph_file, dpi=150, bbox_inches='tight', facecolor='white')
            plt.close()

            logger.info(f"  [Graph] ✓ Saved: {graph_file.name}")

        except Exception as e:
            logger.error(f"  [Graph] Error: {e}")
            plt.close()


In [11]:
# GENERIC THEME EXTRACTOR

class ThemeExtractor:
    """Estrattore temi generici usando clustering semantico"""

    def __init__(self, embedding_model: SentenceTransformer):
        self.embedding_model = embedding_model

    def extract_themes(
        self,
        keywords: List[str],
        laws: List[str],
        document_text: str
    ) -> List[str]:
        """Estrae temi principali da keyword e leggi"""

        if len(keywords) < 3:
            themes = [kw.capitalize() for kw in keywords[:3]]
            if laws:
                themes.append("Legislation")
            return themes

        # Cluster keyword con embeddings
        embeddings = self.embedding_model.encode(keywords)
        n_clusters = min(4, max(2, len(keywords) // 3))

        try:
            clustering = AgglomerativeClustering(
                n_clusters=n_clusters,
                metric='cosine',
                linkage='average'
            )
            labels = clustering.fit_predict(embeddings)
        except:
            return [kw.capitalize() for kw in keywords[:4]]

        themes = []
        clusters = defaultdict(list)
        for i, label in enumerate(labels):
            clusters[label].append(keywords[i])

        for cluster_kw in clusters.values():
            if len(cluster_kw) == 1:
                themes.append(cluster_kw[0].capitalize())
            elif len(cluster_kw) == 2:
                themes.append(f"{cluster_kw[0].capitalize()} and {cluster_kw[1]}")
            else:
                # Trova keyword più rappresentativa (più vicina al centroid)
                cluster_embs = self.embedding_model.encode(cluster_kw)
                centroid = np.mean(cluster_embs, axis=0)

                distances = [
                    (kw, float(cosine_similarity([emb], [centroid])[0][0]))
                    for kw, emb in zip(cluster_kw, cluster_embs)
                ]
                distances.sort(key=lambda x: x[1], reverse=True)

                top2 = distances[:2]
                themes.append(f"{top2[0][0].capitalize()} and {top2[1][0]}")

        # Aggiungi tema "Legislation" se ci sono molte leggi
        if laws and len(laws) >= 2:
            if not any("legislat" in t.lower() for t in themes):
                themes.append("Legislation")

        return themes[:4]

In [12]:
# ENHANCED CATEGORY CLASSIFIER CON DOMAIN TERMS

class CategoryClassifier:
    """Classificatore categorie con domain terms per OSHA"""

    def __init__(self, embedding_model: SentenceTransformer, valid_categories: List[str], category_terms: Dict[str, List[str]]):
        self.embedding_model = embedding_model
        self.valid_categories = valid_categories
        self.category_terms = category_terms
        self.category_embeddings = embedding_model.encode(valid_categories)

    def classify(self, keywords: List[str], themes: List[str], laws: List[str]) -> str:
        """Classifica il documento in una categoria OSH"""

        all_text = ' '.join(keywords + themes).lower()

        # LIVELLO 1: Scoring basato su domain terms
        domain_scores = {}
        for category, terms in self.category_terms.items():
            matches = sum(1 for term in terms if term in all_text)
            domain_scores[category] = matches / max(len(terms), 1)

        # Aggiungi score per keyword overlap generico
        for category in self.valid_categories:
            if category not in domain_scores:
                cat_words = set(category.lower().split())
                text_words = set(all_text.split())
                overlap = len(cat_words & text_words)
                domain_scores[category] = overlap / max(len(cat_words), 1)

        # Top 5 candidates
        top_candidates = sorted(domain_scores.items(), key=lambda x: x[1], reverse=True)[:5]

        # LIVELLO 2: Semantic similarity
        description = f"""Keywords: {', '.join(keywords[:10])}.
Themes: {', '.join(themes[:3])}.
Laws: {', '.join(laws[:3]) if laws else 'General OSH'}."""

        desc_emb = self.embedding_model.encode([description])

        best_category = None
        best_score = 0

        for category, domain_score in top_candidates:
            cat_idx = self.valid_categories.index(category)
            sem_score = float(cosine_similarity(desc_emb, [self.category_embeddings[cat_idx]])[0][0])

            # Combina: 70% domain + 30% semantic
            combined_score = 0.7 * domain_score + 0.3 * sem_score

            if combined_score > best_score:
                best_score = combined_score
                best_category = category

        # Fallback a "Other"
        if best_score < Config.CATEGORY_THRESHOLD:
            return "Other"

        return best_category

In [13]:
# IMPROVED PROFESSION EXTRACTOR (CON BLACKLIST)

class ImprovedProfessionExtractor:
    def __init__(self, embedding_model: SentenceTransformer, known_professions: List[str], blacklist: Set[str]):
        self.embedding_model = embedding_model
        self.known_professions = known_professions
        self.blacklist = blacklist

        print("[Professions] Pre-computing embeddings...")
        self.profession_embeddings = self.embedding_model.encode(known_professions)
        print("[Professions] ✓ Pronto")

    def extract_professions(self,
                            full_text: str,
                            keywords: List[str],
                            themes: List[str],
                            category: str,
                            top_k: int = 3,
                            threshold: float = 0.50) -> List[str]:  # 0.50 per più recall

        doc_desc = f"""Settore: {category}.
Parole chiave: {', '.join(keywords[:8])}.
Temi: {', '.join(themes[:3])}.
Contenuto: {full_text[:1500]}"""  # Fix: full_text non document_text

        doc_emb = self.embedding_model.encode([doc_desc])
        similarities = cosine_similarity(doc_emb, self.profession_embeddings)[0]

        ranked = sorted(  # Fix: ranked non professions_sim
            zip(self.known_professions, similarities),
            key=lambda x: x[1],
            reverse=True
        )

        # Filtra blacklist e applica threshold
        selected = []
        prev_emb = None  # Fix: indentazione corretta
        for prof, sim in ranked:  # Fix: ranked
            if sim < threshold:
                break
            if prof in self.blacklist:  # Fix: self.blacklist
                continue
            # Diversity: skip se sim >0.7 con precedente
            if prev_emb is not None:
                prof_emb = self.embedding_model.encode([prof])[0]
                sim_prev = float(cosine_similarity([prof_emb], [prev_emb])[0][0])
                if sim_prev > 0.70:
                    continue
            selected.append(prof)
            prev_emb = self.embedding_model.encode([prof])[0]  # Fix: indentazione
            if len(selected) >= top_k:
                break

        # Fallback se 0 found: infer da category (non generici, evita blacklist)
        if not selected:
            fallback_map = {
                "Rischio chimico e cancerogeno": ["Lavoratori esposti ad agenti chimici pericolosi"],
                "Rischio biologico": ["Lavoratori esposti ad agenti biologici"],
                "Infortuni e malattie professionali": ["Tecnici della prevenzione"],
                "Altro": ["Tecnici della prevenzione"],  # Non "Lavoratori generici" (blacklist)
                "Salute alimentare e allergie occupazionali": ["Medici del lavoro per allergie"],  # Se espandi category
                "Inclusione e disabilità sul lavoro": ["Coordinatori per la sicurezza"],  # Esempio
                "Eventi lesivi e analisi statistica": ["Tecnici della prevenzione"]  # Esempio
            }
            selected = fallback_map.get(category, ["Lavoratori esposti a rumore e vibrazioni"])  # Default non-blacklist

        return selected[:top_k]

In [14]:
# SUMMARY GENERATOR

class OllamaClient:
    # Client per interazione con LLM esterno (Ollama)
    def __init__(self, model: str = "llama3.1:8b"):
        self.model = model
        self.base_url = "http://localhost:11434"

    def generate(self, prompt: str, max_tokens: int = 150) -> Optional[str]:
        try:
            # Import requests è necessario. Assunto che sia installato nell'ambiente.
            response = requests.post(
                f"{self.base_url}/api/generate",
                json={
                    "model": self.model,
                    "prompt": prompt,
                    "stream": False,
                    "temperature": 0.1,
                    "max_tokens": max_tokens
                },
                timeout=120
            )
            return response.json()['response'].strip() if response.status_code == 200 else None
        except Exception as e:
            # Fallback in caso di errore di connessione o LLM non disponibile
            return None


class ReliableSummaryGenerator:

    def __init__(self, llm_client: Optional[OllamaClient], embedding_model: SentenceTransformer, use_llm: bool = False):
        self.llm = llm_client
        self.embedding_model = embedding_model
        self.use_llm = use_llm

    def generate_with_confidence(
        self,
        document_text: str,
        keywords: List[str],
        laws: List[str],
        category: str,
        themes: List[str]
    ) -> Tuple[str, Dict[str, float]]:

        if not self.use_llm or not self.llm:
            summary = self._create_structured_summary(keywords, laws, category, themes, document_text)
        else:
            # Prova LLM
            prompt = self._create_prompt(document_text, keywords, laws, category, themes)
            llm_summary = self.llm.generate(prompt, max_tokens=120)

            if llm_summary and len(llm_summary.split()) >= 15 and len(llm_summary.split()) <= 60:
                summary = self._clean_summary(llm_summary)

                if self._has_obvious_errors(summary):
                    summary = self._create_structured_summary(keywords, laws, category, themes, document_text)
            else:
                summary = self._create_structured_summary(keywords, laws, category, themes, document_text)

        confidence = self._calculate_confidence(summary, document_text, keywords, laws)

        return summary, confidence

    def _create_prompt(self, text: str, keywords: List[str], laws: List[str], category: str, themes: List[str]) -> str:
        return f"""Scrivi UNA sintesi tecnica di MASSIMO 50 parole per questo documento INAIL.

SETTORE: {category}
TEMI: {', '.join(themes[:3])}
KEYWORDS (usane almeno 4): {', '.join(keywords[:6])}
NORMATIVE: {', '.join(laws[:2]) if laws else 'Non specificate'}

TESTO:
{text[:2500]}

REGOLE:
- Max 50 parole
- Usa almeno 4 keywords
- Cita normative con nomenclatura esatta
- Linguaggio tecnico INAIL
- NO "Il documento...", scrivi direttamente i contenuti

Sintesi:"""

    def _create_structured_summary(
        self,
        keywords: List[str],
        laws: List[str],
        category: str,
        themes: List[str],
        document_text: str
    ) -> str:

        top_keywords = keywords[:min(4, len(keywords))]

        if themes and len(themes) > 0:
            main_theme = themes[0].lower()
            intro = f"Documento tecnico che tratta {main_theme}"
        else:
            intro = f"Documento nel settore {category.lower()}"

        if len(top_keywords) >= 3:
            kw_part = f"con focus su {top_keywords[0]}, {top_keywords[1]} e {top_keywords[2]}"
        elif len(top_keywords) == 2:
            kw_part = f"con focus su {top_keywords[0]} e {top_keywords[1]}"
        elif len(top_keywords) == 1:
            kw_part = f"relativo a {top_keywords[0]}"
        else:
            kw_part = ""

        if laws and len(laws) <= 3:
            law_part = f"Riferimenti normativi: {', '.join(laws)}"
        elif laws and len(laws) > 3:
            law_part = f"Principali normative: {laws[0]}, {laws[1]}"
        else:
            law_part = ""

        summary_parts = [intro]
        if kw_part:
            summary_parts.append(kw_part)

        if len(summary_parts) == 2:
            summary = f"{summary_parts[0]} {summary_parts[1]}"
        else:
            summary = summary_parts[0]

        if law_part:
            summary = f"{summary}. {law_part}"

        if not summary.endswith('.'):
            summary += '.'

        summary = summary[0].upper() + summary[1:]

        return summary

    def _clean_summary(self, summary: str) -> str:

        summary = summary.strip()
        summary = re.sub(r'^(Il documento|Questo documento|La presente|Il presente)\s+', '', summary, flags=re.IGNORECASE)
        summary = re.sub(r'\*\*.*?\*\*', '', summary)

        words = summary.split()
        if len(words) > 55:
            summary = ' '.join(words[:52]) + '...'

        return summary

    def _has_obvious_errors(self, text: str) -> bool:

        errors = [
            'invenzione conoscitiva',
            'suggerimento attività',
            'entre ',
            'comprovando',
            'conforme D',
        ]

        text_lower = text.lower()
        for error in errors:
            if error.lower() in text_lower:
                return True

        return False

    def _calculate_confidence(
        self,
        summary: str,
        document_text: str,
        keywords: List[str],
        laws: List[str]
    ) -> Dict[str, float]:

        try:
            # NESSUNA MODIFICA NECESSARIA: .encode() è corretto per SentenceTransformer
            sum_emb = self.embedding_model.encode([summary])[0]
            doc_emb = self.embedding_model.encode([document_text[:4500]])[0]
            grounding = float(cosine_similarity([sum_emb], [doc_emb])[0][0])
        except:
            grounding = 0.5

        summary_lower = summary.lower()
        kw_mentioned = sum(1 for kw in keywords[:8] if kw.lower() in summary_lower)
        kw_coverage = kw_mentioned / min(len(keywords), 8)

        if laws:
            laws_mentioned = 0
            for law in laws[:4]:
                law_parts = re.findall(r'\d+', law)
                if any(part in summary for part in law_parts):
                    laws_mentioned += 1

            factuality = laws_mentioned / min(len(laws), 4)
        else:
            factuality = 0.7

        overall = 0.50 * grounding + 0.35 * kw_coverage + 0.15 * factuality

        return {
            'overall': round(overall, 3),
            'grounding': round(grounding, 3),
            'keyword_coverage': round(kw_coverage, 3),
            'factuality': round(factuality, 3)
        }

In [15]:
# GLOBAL CONFIDENCE CALCULATOR

class GlobalConfidenceCalculator:
    """
    Calcola confidence globale con gestione adattiva per documenti senza leggi.
    Adattato per OSHA/EU che potrebbe avere meno riferimenti normativi diretti.
    """

    def calculate(
        self,
        keyword_confidence: Dict[str, float],
        law_confidence: Dict[str, float],
        summary_confidence: Dict[str, float]
    ) -> Dict[str, Any]:

        kw_score = keyword_confidence.get('overall', 0.0)
        law_score = law_confidence.get('overall', 0.0)
        sum_score = summary_confidence.get('overall', 0.0)

        # LOGICA ADATTIVA: documenti OSHA/EU hanno meno riferimenti normativi specifici
        num_laws = law_confidence.get('count', 0.0)

        if num_laws == 0 or law_score < 0.15:
            # Documento informativo/tecnico senza normative specifiche
            # Dai più peso a keywords (contenuto tecnico) e summary
            overall = 0.55 * kw_score + 0.20 * law_score + 0.25 * sum_score
            adapted = True
            adaptation_reason = "Technical/informative document without specific legislation references"
        else:
            # Documento con riferimenti normativi: usa pesi standard
            overall = 0.50 * kw_score + 0.25 * law_score + 0.25 * sum_score
            adapted = False
            adaptation_reason = None

        review_reasons = []

        # Soglie ADATTIVE
        if adapted:
            # Soglie più permissive per documenti tecnici
            high_threshold = 0.62
            medium_threshold = 0.45
        else:
            high_threshold = 0.68
            medium_threshold = 0.52

        if overall >= high_threshold:
            level = "HIGH"
            needs_review = False
        elif overall >= medium_threshold:
            level = "MEDIUM"
            needs_review = True
            review_reasons.append("Medium confidence: sample review recommended")
        else:
            level = "LOW"
            needs_review = True
            review_reasons.append("Low confidence: manual review required")

        # Review reasons
        if kw_score < 0.48:
            review_reasons.append(f"Keywords confidence low ({kw_score:.2f})")

        if not adapted and law_score < 0.43:
            review_reasons.append(f"Laws confidence low ({law_score:.2f})")

        if sum_score < 0.48:
            review_reasons.append(f"Summary grounding low ({sum_score:.2f})")

        return {
            'overall_score': round(overall, 3),
            'confidence_level': level,
            'needs_review': needs_review,
            'adapted_scoring': adapted,
            'adaptation_reason': adaptation_reason,
            'components': {
                'keywords': round(kw_score, 3),
                'laws': round(law_score, 3),
                'summary': round(sum_score, 3)
            },
            'detailed_components': {
                'keywords': keyword_confidence,
                'laws': law_confidence,
                'summary': summary_confidence
            },
            'review_reasons': review_reasons
        }

In [16]:
# UTILITY

def extract_full_document_text(document_data: Dict[str, Any]) -> str:
    chunks = []
    abstract = document_data.get('web_metadata', {}).get('abstract', '')
    if abstract:
        chunks.append(f"ABSTRACT:\n{abstract}\n")

    content = document_data.get('document_content', {})
    if content.get('plain_text'):
        chunks.append(content['plain_text'])
    elif content.get('markdown_content'):
        chunks.append(content['markdown_content'])

    return '\n\n'.join(chunks)


def setup_logging(log_dir: Path):
    log_dir.mkdir(exist_ok=True, parents=True)
    log_file = log_dir / f"enrichment_final_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"
    logging.basicConfig(
        level=logging.INFO,
        format='%(asctime)s - %(message)s',
        handlers=[logging.FileHandler(log_file), logging.StreamHandler()]
    )
    return logging.getLogger(__name__), log_file

In [17]:
# DOCUMENT ENRICHMENT

def enrich_document_final(
    document_data: Dict[str, Any],
    law_extractor: EULawExtractor,
    keyword_extractor: DomainAwareTFIDFExtractor,
    theme_extractor: ThemeExtractor,
    profession_extractor: ImprovedProfessionExtractor,
    summary_generator: ReliableSummaryGenerator,
    category_classifier: CategoryClassifier,
    confidence_calculator: GlobalConfidenceCalculator,
    logger: logging.Logger,
    save_graph: bool = True
) -> Optional[Dict[str, Any]]:

    doc_title = document_data.get('web_metadata', {}).get('title', 'N/A')
    logger.info(f"\n{'='*70}")
    logger.info(f"Processing: {doc_title[:60]}")
    logger.info(f"{'='*70}")

    try:
        # ESTRAI TESTO CON FUNZIONE OSHA
        full_text = extract_osha_document_text(document_data)  # ← Usa funzione OSHA

        # CONTROLLO None
        if not full_text or full_text.strip() == "":
            logger.warning("⚠ Empty or None text, skipping")
            return None

        logger.info(f"[Text] {len(full_text)} chars")

        if len(full_text) < Config.MIN_TEXT_LENGTH:
            logger.warning("Text too short, skipping")
            return None

        # 1) LEGGI
        laws, law_confidence = law_extractor.extract_with_confidence(full_text)
        logger.info(f"[Laws] {len(laws)} found | Confidence: {law_confidence['overall']:.3f}")

        # 2) KEYWORDS
        safe_filename = re.sub(r'[^\w\-_]', '_', doc_title[:30]) if doc_title else "unknown"
        graph_path = Config.GRAPH_DIR / f"{safe_filename}.png" if save_graph else None

        keywords, kw_metadata, G = keyword_extractor.extract_keywords_with_confidence(
            full_text, top_n=8, save_graph=save_graph,
            graph_path=graph_path, doc_title=doc_title
        )

        logger.info(f"[Keywords] {len(keywords)} selected | Confidence: {kw_metadata['confidence']['overall']:.3f}")
        for i, kw in enumerate(keywords, 1):
            logger.info(f"  {i}. {kw}")

        # 3) TEMI
        themes = theme_extractor.extract_themes(keywords, laws, full_text)
        logger.info(f"[Themes] {len(themes)} extracted")
        for tema in themes:
            logger.info(f"  • {tema}")

        # 4) CATEGORIA (domain-aware)
        categoria = category_classifier.classify(keywords, themes, laws)
        logger.info(f"[Category] {categoria}")

        # 5) SINTESI (preferisce fallback strutturato)
        sintesi, summary_confidence = summary_generator.generate_with_confidence(
            full_text, keywords, laws, categoria, themes
        )
        logger.info(f"[Summary] Confidence: {summary_confidence['overall']:.3f}")
        logger.info(f"  {sintesi}")

        # 6) PROFESSIONI (con blacklist)
        professioni = profession_extractor.extract_professions(
            full_text, keywords, themes, categoria, top_k=3, threshold=Config.PROFESSION_THRESHOLD
        )
        logger.info(f"[Professions] {len(professioni)} found")
        for prof in professioni:
            logger.info(f"  • {prof}")

        # 7) CONFIDENCE GLOBALE (CON ADATTAMENTO)
        global_confidence = confidence_calculator.calculate(
            kw_metadata['confidence'],
            law_confidence,
            summary_confidence
        )

        logger.info(f"\n[GLOBAL CONFIDENCE]")
        logger.info(f"  Overall: {global_confidence['overall_score']:.3f}")
        logger.info(f"  Level: {global_confidence['confidence_level']}")

        # LOG SCORING ADATTIVO
        if global_confidence.get('adapted_scoring'):
            logger.info(f"  ⚙ Adaptive Scoring: {global_confidence.get('adaptation_reason')}")

        logger.info(f"  Needs Review: {'YES' if global_confidence['needs_review'] else 'NO'}")

        if global_confidence['review_reasons']:
            logger.info(f"  Reasons:")
            for reason in global_confidence['review_reasons']:
                logger.info(f"    ⚠ {reason}")

        # 8) METADATA
        metadata = {
            'sintesi': sintesi,
            'categoria_principale': categoria,
            'articoli_legge': laws,
            'categorie_professionali': professioni,
            'parole_chiave': keywords,
            'temi_principali': themes,
            'confidence': global_confidence,
            'extraction_details': kw_metadata,
            'extraction_methods': {
                'keywords': 'TF-IDF Domain-Aware + EmbedRank (Bennani-Smires 2018)',
                'themes': 'Semantic Clustering',
                'category': 'Domain-Aware Hierarchical (term mapping + semantic)',
                'laws': 'Regex Pattern Matching + Multi-metric',
                'summary': 'Structured Fallback (deterministic) + optional LLM',
                'professions': 'BERT Semantic + Blacklist Filtering',
                'confidence': 'Multi-component (distribution + coverage + specificity + coherence)'
            },
            'generated_at': datetime.now().isoformat(),
            'version': 'Final_EmbeddingGemma_v4.0_AdaptiveScoring_OSHA'
        }

        enriched_doc = document_data.copy()
        enriched_doc['semantic_metadata'] = metadata

        logger.info(f"{'='*70}\n")

        return enriched_doc

    except Exception as e:
        logger.error(f"ERROR processing {doc_title}: {e}")
        traceback.print_exc()
        return None


In [18]:
# OLLAMA SETUP

def check_ollama_installed():
    try:
        result = subprocess.run(['which', 'ollama'], capture_output=True, timeout=5)
        return result.returncode == 0
    except:
        return False


def install_ollama():
    if check_ollama_installed():
        print("[Setup] Ollama già installato ✓")
        return True

    print("[Setup] Installing Ollama...")
    try:
        subprocess.run(['bash', '-c', 'curl -fsSL https://ollama.ai/install.sh | sh'],
                       check=True, capture_output=True, timeout=300)
        print("✓ Ollama installed")
        return True
    except Exception as e:
        print(f"✗ Error: {e}")
        return False


def start_ollama():
    print("[Setup] Starting Ollama...")
    try:
        # Check se già running
        result = subprocess.run(['pgrep', '-x', 'ollama'], capture_output=True)
        if result.returncode == 0:
            print("✓ Ollama già running")
            return True

        # Usa Popen per avviare in background
        subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
        time.sleep(5) # Give time to start
        print("✓ Server started")
        return True
    except Exception as e:
        print(f"✗ Error starting ollama: {e}")
        return False


def pull_model(model_name: str):
    print(f"[Setup] Checking {model_name}...")
    try:
        # Check se già scaricato
        result = subprocess.run(['ollama', 'list'], capture_output=True, text=True, timeout=10)
        if model_name.split(':')[0] in result.stdout:
            print(f"✓ {model_name} già disponibile")
            return True

        print(f"[Setup] Downloading {model_name}...")
        result = subprocess.run(['ollama', 'pull', model_name],
                                capture_output=True, timeout=600, text=True)
        if result.returncode == 0:
            print(f"✓ {model_name} ready")
            return True
        else:
            print(f"✗ Download failed: {result.stderr}")
            return False
    except Exception as e:
        print(f"✗ Error: {e}")
        return False

In [19]:
# QUICK TEST & BATCH PROCESSING

def quick_test():
    # Versione modificata con EmbeddingGemma per OSHA
    print("\n" + "="*80)
    print("OSHA - FINAL CORRECTED ENRICHMENT SYSTEM (EmbeddingGemma)")
    print("="*80 + "\n")

    print("[Setup] Verifying paths...")
    if not Config.setup():
        print("\n✗ Setup failed\n")
        return

    logger, log_file = setup_logging(Config.LOG_DIR)

    # LLM Setup
    llm_client = None
    if Config.USE_LLM_SUMMARY:
        print("\n[Setup] Configuring LLM...")
        if not install_ollama() or not start_ollama():
            print("✗ Ollama setup failed, using fallback summary")
        elif pull_model(Config.OLLAMA_MODEL):
            llm_client = OllamaClient(model=Config.OLLAMA_MODEL)
        else:
            print("✗ Model download failed, using fallback summary")
    else:
        print("\n[Setup] Using structured fallback summary (no LLM)")

    # EMBEDDING MODEL
    print("\n[Setup] Initializing embedding model...")
    embedding_model = load_embedding_model(use_gemma=Config.USE_EMBEDDING_GEMMA)
    print("✓ Embedding model ready\n")

    # Inizializza componenti
    law_extractor = EULawExtractor()
    keyword_extractor = DomainAwareTFIDFExtractor(embedding_model, Config.DOMAIN_TERMS)
    theme_extractor = ThemeExtractor(embedding_model)
    profession_extractor = ImprovedProfessionExtractor(
        embedding_model,
        Config.KNOWN_PROFESSIONS,
        Config.PROFESSION_BLACKLIST
    )
    summary_generator = ReliableSummaryGenerator(
        llm_client,
        embedding_model,
        Config.USE_LLM_SUMMARY
    )
    category_classifier = CategoryClassifier(
        embedding_model,
        Config.DOCUMENT_CATEGORIES,
        Config.CATEGORY_TERMS
    )
    confidence_calculator = GlobalConfidenceCalculator()

    print("✓ All components ready\n")

    json_files = sorted([f for f in Config.JSON_DIR.glob("*.json")
                         if not f.name.startswith('vector_db')])

    if not json_files:
        print(f"✗ No JSON in {Config.JSON_DIR}")
        return

    print(f"Available: {len(json_files)}\n")
    for i, f in enumerate(json_files[:10], 1):
        print(f"{i}. {f.name[:70]}")

    idx_input = input(f"\nWhich? (1-{min(10, len(json_files))}, default 1): ").strip()
    idx = int(idx_input) - 1 if idx_input.isdigit() else 0
    test_file = json_files[idx] if 0 <= idx < len(json_files) else json_files[0]

    save_graph = input("Save EmbedRank graph? (y/n, default n): ").strip().lower() == 'y'

    with open(test_file, 'r', encoding='utf-8') as f:
        doc = json.load(f)

    title = doc.get('web_metadata', {}).get('title', 'N/A')

    print(f"\n{'='*80}")
    print(f"TEST DOCUMENT")
    print(f"{'='*80}")
    print(f"Title: {title}")
    print(f"{'='*80}\n")

    enriched = enrich_document_final(
        doc, law_extractor, keyword_extractor, theme_extractor,
        profession_extractor, summary_generator, category_classifier,
        confidence_calculator, logger, save_graph=save_graph
    )

    if enriched:
        meta = enriched['semantic_metadata']
        conf = meta['confidence']

        print(f"\n{'='*80}")
        print("ENRICHMENT RESULTS - FINAL CORRECTED")
        print(f"{'='*80}\n")

        print(f"SUMMARY:")
        print(f"  {meta['sintesi']}\n")

        print(f"CATEGORY: {meta['categoria_principale']}\n")

        print(f"LAWS ({len(meta['articoli_legge'])}):")
        for law in meta['articoli_legge'][:8]:
            print(f"  • {law}")
        print()

        print(f"KEYWORDS ({len(meta['parole_chiave'])}):")
        for i, kw in enumerate(meta['parole_chiave'], 1):
            print(f"  {i}. {kw}")
        print()

        print(f"THEMES ({len(meta['temi_principali'])}):")
        for tema in meta['temi_principali']:
            print(f"  • {tema}")
        print()

        if meta.get('categorie_professionali'):
            print(f"PROFESSIONS ({len(meta['categorie_professionali'])}):")
            for prof in meta['categorie_professionali']:
                print(f"  • {prof}")
            print()

        print(f"{'='*80}")
        print("CONFIDENCE ANALYSIS")
        print(f"{'='*80}")
        print(f"Overall: {conf['overall_score']:.3f}")
        print(f"Level: {conf['confidence_level']}")

        # Mostra scoring adattivo se presente
        if conf.get('adapted_scoring'):
            print(f"⚙ Adaptive: {conf.get('adaptation_reason')}")

        print(f"Review: {'YES ⚠' if conf['needs_review'] else 'NO ✓'}\n")

        print("Components:")
        for component, score in conf['components'].items():
            bar = '█' * int(score * 20)
            print(f"  {component.capitalize():12s}: {bar:20s} {score:.3f}")

        if conf['review_reasons']:
            print(f"\nReview Reasons:")
            for reason in conf['review_reasons']:
                print(f"  • {reason}")

        print(f"\n{'='*80}\n")

        test_path = Config.ENRICHED_DIR / f"TEST_FINAL_{test_file.name}"
        with open(test_path, 'w', encoding='utf-8') as f:
            json.dump(enriched, f, ensure_ascii=False, indent=2)

        print(f"✓ Saved: {test_path.name}")
        print(f"✓ Log: {log_file.name}\n")

        if save_graph:
            print(f"✓ Graph saved in: {Config.GRAPH_DIR}\n")

    else:
        print("✗ ENRICHMENT FAILED\n")


def batch_process_all():
    # Versione batch con EmbeddingGemma per OSHA
    print("\n" + "="*80)
    print("BATCH PROCESSING - ALL DOCUMENTS (EmbeddingGemma Edition)")
    print("="*80 + "\n")

    if not Config.setup():
        print("✗ Setup failed")
        return

    logger, log_file = setup_logging(Config.LOG_DIR)

    # LLM setup
    llm_client = None
    if Config.USE_LLM_SUMMARY:
        if install_ollama() and start_ollama() and pull_model(Config.OLLAMA_MODEL):
            llm_client = OllamaClient(model=Config.OLLAMA_MODEL)
        else:
            print("⚠ LLM setup failed, using fallback summary\n")
    else:
        print("Using structured fallback summary (no LLM)\n")

    # EMBEDDING MODEL
    print("[Setup] Initializing embedding model...")
    embedding_model = load_embedding_model(use_gemma=Config.USE_EMBEDDING_GEMMA)
    print("✓ Ready\n")

    # Initialize components
    law_extractor = EULawExtractor()
    keyword_extractor = DomainAwareTFIDFExtractor(embedding_model, Config.DOMAIN_TERMS)
    theme_extractor = ThemeExtractor(embedding_model)
    profession_extractor = ImprovedProfessionExtractor(
        embedding_model,
        Config.KNOWN_PROFESSIONS,
        Config.PROFESSION_BLACKLIST
    )
    summary_generator = ReliableSummaryGenerator(
        llm_client,
        embedding_model,
        Config.USE_LLM_SUMMARY
    )
    category_classifier = CategoryClassifier(
        embedding_model,
        Config.DOCUMENT_CATEGORIES,
        Config.CATEGORY_TERMS
    )
    confidence_calculator = GlobalConfidenceCalculator()

    # Get all JSON files
    json_files = sorted([f for f in Config.JSON_DIR.glob("*.json")
                         if not f.name.startswith('vector_db')])

    print(f"Found {len(json_files)} documents\n")

    success_count = 0
    fail_count = 0
    confidence_stats = {'HIGH': 0, 'MEDIUM': 0, 'LOW': 0}

    for i, json_file in enumerate(json_files, 1):
        print(f"\n[{i}/{len(json_files)}] Processing: {json_file.name[:60]}")

        try:
            with open(json_file, 'r', encoding='utf-8') as f:
                doc = json.load(f)

            enriched = enrich_document_final(
                doc, law_extractor, keyword_extractor, theme_extractor,
                profession_extractor, summary_generator, category_classifier,
                confidence_calculator, logger, save_graph=True  # ABILITA GRAFI
            )

            if enriched:
                output_path = Config.ENRICHED_DIR / json_file.name
                with open(output_path, 'w', encoding='utf-8') as f:
                    json.dump(enriched, f, ensure_ascii=False, indent=2)

                conf_level = enriched['semantic_metadata']['confidence']['confidence_level']
                confidence_stats[conf_level] += 1

                success_count += 1
                print(f"  ✓ Saved | Confidence: {conf_level}")
            else:
                fail_count += 1
                print(f"  ✗ Failed (Text too short or enrichment error)")

        except Exception as e:
            fail_count += 1
            print(f"  ✗ Error: {e}")
            logger.error(f"FATAL ERROR processing {json_file.name}: {e}")

    print(f"\n{'='*80}")
    print(f"BATCH COMPLETE")
    print(f"{'='*80}")
    print(f"Success: {success_count}")
    print(f"Failed: {fail_count}")
    print(f"Total: {len(json_files)}")
    print(f"\nConfidence Distribution:")
    total_success = max(success_count, 1)
    print(f"  HIGH:   {confidence_stats['HIGH']} ({confidence_stats['HIGH']/total_success*100:.1f}%)")
    print(f"  MEDIUM: {confidence_stats['MEDIUM']} ({confidence_stats['MEDIUM']/total_success*100:.1f}%)")
    print(f"  LOW:    {confidence_stats['LOW']} ({confidence_stats['LOW']/total_success*100:.1f}%)")
    print(f"{'='*80}\n")

In [20]:
# ENTRY POINT
if __name__ == "__main__":
    print("\n" + "="*80)
    print("OSHA/EU-OSHA - FINAL CORRECTED ENRICHMENT SYSTEM (EmbeddingGemma)")
    print("="*80)

    mode = input("Mode? [1] Quick test  [2] Batch all (default 1): ").strip()

    if mode == "2":
        batch_process_all()
    else:
        quick_test()


OSHA/EU-OSHA - FINAL CORRECTED ENRICHMENT SYSTEM (EmbeddingGemma)
Mode? [1] Quick test  [2] Batch all (default 1): 2

BATCH PROCESSING - ALL DOCUMENTS (EmbeddingGemma Edition)

✓ Trovati 1065 file JSON
✓ Trovati 864 file con keyword pre-estratte OSHA
[Setup] Installing Ollama...
✓ Ollama installed
[Setup] Starting Ollama...
✓ Server started
[Setup] Checking llama3.1:8b...
[Setup] Downloading llama3.1:8b...
✓ llama3.1:8b ready
[Setup] Initializing embedding model...

[Embedding Model] Tentativo caricamento OFFLINE forzato da: /content/drive/MyDrive/INAIL_Thesis_Data_old/EmbeddingGemma_Offline
✓ Modello EmbeddingGemma locale caricato con successo (HPC ready)
  Device: cpu
✓ Ready

[TF-IDF+EmbedRank] Inizializzazione per OSH (English)...
[TF-IDF+EmbedRank] Modello attivo: unknown
[TF-IDF+EmbedRank] ✓ Pronto per elaborazione inglese
[Professions] Pre-computing embeddings...
[Professions] ✓ Pronto
Found 1065 documents


[1/1065] Processing: 2013_Annual_Management_Plan___Work_Programme_2025

  ✗ Failed (Text too short or enrichment error)

[874/1065] Processing: Summary_-_Digital_platform_work_and_occupational_s_20251210_
  ✓ Saved | Confidence: HIGH

[875/1065] Processing: Summary_-_Digital_technologies_at_work_and_psychos_20251204_
  ✓ Saved | Confidence: HIGH

[876/1065] Processing: Summary_-_Digital_technologies_for_worker_manageme_20251204_
  ✓ Saved | Confidence: HIGH

[877/1065] Processing: Summary_-_Education___evidence_from_the_European_S_20251205_
  ✓ Saved | Confidence: HIGH

[878/1065] Processing: Summary_-_Exposure_to_carcinogens_and_work-related_20251210_
  ✓ Saved | Confidence: HIGH

[879/1065] Processing: Summary_-_Feasibility_study_on_the_development_of__20251210_
  ✓ Saved | Confidence: HIGH

[880/1065] Processing: Summary_-_Foresight_on_new_and_emerging_occupation_20251210_
  ✓ Saved | Confidence: HIGH

[881/1065] Processing: Summary_-_Foresight_study_on_the_circular_economy__20251205_
  ✓ Saved | Confidence: HIGH

[882/1065] Processing: Summary_-_German

In [ ]:
MY_TOKEN = os.environ.get("HUGGINGFACE_TOKEN", "")

In [ ]:
# ================================================================================
# TEST PESI KEYWORDS - OSHA EDITION
# ================================================================================
# Test su 50 documenti OSHA per trovare pesi ottimali keyword confidence
# Adattato per: OSHADomainAwareTFIDFExtractor + extract_osha_document_text
# ================================================================================

import pandas as pd
from datetime import datetime

def test_keyword_weights_osha(num_docs: int = 50) -> dict:
    """
    Testa combinazioni di pesi per keyword confidence su documenti OSHA.
    Componenti: [score_distribution, document_coverage, term_specificity, graph_coherence]
    """
    combinations = [
        [0.30, 0.30, 0.25, 0.15],  # Originale bilanciato
        [0.25, 0.35, 0.25, 0.15],  # Più coverage (OSHA tecnici)
        [0.35, 0.25, 0.20, 0.20],  # Più distribution + coherence
        [0.20, 0.40, 0.20, 0.20],  # Max coverage/specificity
        [0.28, 0.32, 0.22, 0.18],  # Bilanciato variante
        [0.15, 0.45, 0.25, 0.15]   # Max coverage (docs internazionali)
    ]

    json_files = sorted(list(Config.JSON_DIR.glob("*.json")))[:num_docs]
    print(f"\n{'='*80}")
    print(f"TEST KEYWORD WEIGHTS - OSHA")
    print(f"{'='*80}")
    print(f"Testing {len(json_files)} documents\n")

    # Setup embedding model
    print("[Setup] Loading embedding model...")
    embedding_model = load_embedding_model(use_gemma=Config.USE_EMBEDDING_GEMMA)

    # Usa DomainAwareTFIDFExtractor
    keyword_extractor = DomainAwareTFIDFExtractor(embedding_model, Config.DOMAIN_TERMS)
    print("✓ Ready\n")

    results = {}

    for i, combo in enumerate(combinations, 1):
        print(f"[Combo {i}/6] Weights: {combo}")
        total_score = 0
        valid_docs = 0

        for j, json_file in enumerate(json_files):
            try:
                with open(json_file, 'r', encoding='utf-8') as f:
                    doc = json.load(f)

                # Usa extract_osha_document_text
                full_text = extract_osha_document_text(doc)

                if not full_text or len(full_text) < Config.MIN_TEXT_LENGTH:
                    continue

                # Estrazione keyword components
                tfidf_kw = keyword_extractor._extract_tfidf_domain_aware(full_text, top_n=30)
                if not tfidf_kw:
                    continue

                embedrank_kw, G = keyword_extractor._embed_rank(tfidf_kw, full_text)
                final_kw = keyword_extractor._combine_and_diversify(tfidf_kw, embedrank_kw, top_n=8)

                # Calcola confidence originale
                orig_conf = keyword_extractor._calculate_confidence(
                    final_kw, tfidf_kw, embedrank_kw, full_text, G
                )

                # Componenti per test
                components = [
                    orig_conf['score_distribution'],
                    orig_conf['document_coverage'],
                    orig_conf['term_specificity'],
                    orig_conf['graph_coherence']
                ]

                # Score con pesi custom
                custom_overall = sum(w * c for w, c in zip(combo, components))
                total_score += custom_overall
                valid_docs += 1

            except Exception as e:
                continue

        avg_score = total_score / max(valid_docs, 1)
        results[f"Combo_{i}"] = {
            'weights': combo,
            'avg_confidence': round(avg_score, 3),
            'valid_docs': valid_docs
        }
        print(f"  → Avg confidence: {avg_score:.3f} (valid: {valid_docs}/{num_docs})\n")

    # Risultati finali
    print(f"\n{'='*80}")
    print("RESULTS - KEYWORD WEIGHTS")
    print(f"{'='*80}\n")

    # Migliore combinazione
    best_key = max(results, key=lambda k: results[k]['avg_confidence'])
    best = results[best_key]

    print(f"BEST: {best_key}")
    print(f"   Confidence: {best['avg_confidence']:.3f}")
    print(f"   Weights: {best['weights']}")
    print(f"   Valid docs: {best['valid_docs']}/{num_docs}\n")

    print("IMPLEMENTATION:")
    print(f"   In DomainAwareTFIDFExtractor._calculate_confidence():")
    print(f"   overall = {best['weights'][0]} * score_distribution + \\")
    print(f"             {best['weights'][1]} * document_coverage + \\")
    print(f"             {best['weights'][2]} * term_specificity + \\")
    print(f"             {best['weights'][3]} * graph_coherence\n")

    # Tabella comparativa
    print("="*80)
    print("| Combo | Distribution | Coverage | Specificity | Coherence | Avg Conf | Valid |")
    print("|-------|--------------|----------|-------------|-----------|----------|-------|")
    for name, data in results.items():
        w = data['weights']
        print(f"| {name:7s} | {w[0]:12.2f} | {w[1]:8.2f} | {w[2]:11.2f} | {w[3]:9.2f} | {data['avg_confidence']:8.3f} | {data['valid_docs']:5d} |")
    print("="*80 + "\n")

    # Salva CSV
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    df = pd.DataFrame(results).T
    csv_path = Config.LOG_DIR / f"keyword_weights_osha_{timestamp}.csv"
    df.to_csv(csv_path)
    print(f"✓ CSV saved: {csv_path}\n")

    return results

In [ ]:
# ================================================================================
# TEST PESI GLOBAL CONFIDENCE - OSHA EDITION
# ================================================================================

def test_global_weights_osha(num_docs: int = 30) -> dict:
    """
    Testa combinazioni di pesi per global confidence su documenti OSHA.
    Componenti: [keywords, laws, summary]
    NOTA: Con scoring adattivo, alcuni docs avranno pesi diversi automaticamente.
    """
    combinations = [
        [0.45, 0.30, 0.25],  # Originale (pre-adaptive)
        [0.50, 0.25, 0.25],  # Più keywords
        [0.35, 0.35, 0.30],  # Bilanciato
        [0.40, 0.30, 0.30],  # Più summary
        [0.42, 0.33, 0.25],  # Variante grid
        [0.55, 0.20, 0.25]   # Max keywords (OSHA tecnici senza normative)
    ]

    json_files = sorted(list(Config.JSON_DIR.glob("*.json")))[:num_docs]
    print(f"\n{'='*80}")
    print(f"TEST GLOBAL WEIGHTS - OSHA")
    print(f"{'='*80}")
    print(f"Testing {len(json_files)} documents (full enrichment)")
    print(f"⚠ NOTE: Adaptive scoring may override weights for low-law docs\n")

    # Setup components
    print("[Setup] Initializing components...")
    embedding_model = load_embedding_model(use_gemma=Config.USE_EMBEDDING_GEMMA)

    # Usa classi corrette OSHA
    law_extractor = EULawExtractor()
    keyword_extractor = DomainAwareTFIDFExtractor(embedding_model, Config.DOMAIN_TERMS)
    theme_extractor = ThemeExtractor(embedding_model)
    profession_extractor = ImprovedProfessionExtractor(
        embedding_model,
        Config.KNOWN_PROFESSIONS,
        Config.PROFESSION_BLACKLIST
    )
    summary_generator = ReliableSummaryGenerator(None, embedding_model, use_llm=False)
    category_classifier = CategoryClassifier(
        embedding_model,
        Config.DOCUMENT_CATEGORIES,
        Config.CATEGORY_TERMS
    )
    confidence_calculator = GlobalConfidenceCalculator()

    logger_test = logging.getLogger("test_global")
    logger_test.setLevel(logging.WARNING)  # Solo warning per non intasare
    print("✓ Ready\n")

    results = {}

    for i, combo in enumerate(combinations, 1):
        print(f"[Combo {i}/6] Weights: {combo}")
        total_score = 0
        valid_docs = 0
        adaptive_count = 0

        for j, json_file in enumerate(json_files):
            try:
                with open(json_file, 'r', encoding='utf-8') as f:
                    doc = json.load(f)

                # Full enrichment
                enriched = enrich_document_final(
                    doc, law_extractor, keyword_extractor, theme_extractor,
                    profession_extractor, summary_generator, category_classifier,
                    confidence_calculator, logger_test, save_graph=False  # No graphs per velocità
                )

                if enriched:
                    conf = enriched['semantic_metadata']['confidence']

                    # Conta adaptive scoring
                    if conf.get('adapted_scoring'):
                        adaptive_count += 1

                    # Componenti
                    components = [
                        conf['components']['keywords'],
                        conf['components']['laws'],
                        conf['components']['summary']
                    ]

                    # Score custom (solo se NON adaptive, altrimenti usa quello del sistema)
                    if not conf.get('adapted_scoring'):
                        custom_overall = sum(w * c for w, c in zip(combo, components))
                    else:
                        # Per docs adaptive, usa lo score originale
                        custom_overall = conf['overall_score']

                    total_score += custom_overall
                    valid_docs += 1

            except Exception as e:
                continue

        avg_score = total_score / max(valid_docs, 1)
        results[f"Combo_{i}"] = {
            'weights': combo,
            'avg_confidence': round(avg_score, 3),
            'valid_docs': valid_docs,
            'adaptive_docs': adaptive_count
        }
        print(f"  → Avg confidence: {avg_score:.3f} (valid: {valid_docs}/{num_docs}, adaptive: {adaptive_count})\n")

    # Risultati finali
    print(f"\n{'='*80}")
    print("RESULTS - GLOBAL WEIGHTS")
    print(f"{'='*80}\n")

    # Migliore
    best_key = max(results, key=lambda k: results[k]['avg_confidence'])
    best = results[best_key]

    print(f"BEST: {best_key}")
    print(f"   Confidence: {best['avg_confidence']:.3f}")
    print(f"   Weights: {best['weights']}")
    print(f"   Valid docs: {best['valid_docs']}/{num_docs}")
    print(f"   Adaptive docs: {best['adaptive_docs']} (used system weights)\n")

    print("IMPLEMENTATION:")
    print(f"   In GlobalConfidenceCalculator.calculate() (non-adaptive branch):")
    print(f"   overall = {best['weights'][0]} * kw_score + \\")
    print(f"             {best['weights'][1]} * law_score + \\")
    print(f"             {best['weights'][2]} * sum_score\n")

    # Tabella
    print("="*80)
    print("| Combo | Keywords | Laws | Summary | Avg Conf | Valid | Adaptive |")
    print("|-------|----------|------|---------|----------|-------|----------|")
    for name, data in results.items():
        w = data['weights']
        print(f"| {name:7s} | {w[0]:8.2f} | {w[1]:4.2f} | {w[2]:7.2f} | {data['avg_confidence']:8.3f} | {data['valid_docs']:5d} | {data['adaptive_docs']:8d} |")
    print("="*80 + "\n")

    # CSV
    timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
    df = pd.DataFrame(results).T
    csv_path = Config.LOG_DIR / f"global_weights_osha_{timestamp}.csv"
    df.to_csv(csv_path)
    print(f"✓ CSV saved: {csv_path}\n")

    return results

In [ ]:
# ================================================================================
# MENU TEST WEIGHTS - OSHA
# ================================================================================

def run_weight_tests():
    print("\n" + "="*80)
    print("OSHA - WEIGHT OPTIMIZATION TEST")
    print("="*80)
    print("\nOptions:")
    print("  [k] Keyword weights (50 docs, fast)")
    print("  [g] Global weights (30 docs, slower)")
    print("  [b] Both tests (sequential)")
    print("  [q] Quit")

    choice = input("\nChoice (default q): ").strip().lower()

    if choice == 'k':
        test_keyword_weights_osha(50)
    elif choice == 'g':
        test_global_weights_osha(30)
    elif choice == 'b':
        print("\nRunning both tests...\n")
        test_keyword_weights_osha(50)
        print("\n" + "="*80 + "\n")
        test_global_weights_osha(30)
    else:
        print("Exiting.")

In [ ]:
# Run
if __name__ == "__main__":
    run_weight_tests()


OSHA - WEIGHT OPTIMIZATION TEST

Options:
  [k] Keyword weights (50 docs, fast)
  [g] Global weights (30 docs, slower)
  [b] Both tests (sequential)
  [q] Quit

Choice (default q): k

TEST KEYWORD WEIGHTS - OSHA
Testing 50 documents

[Setup] Loading embedding model...

[Embedding Model] Tentativo caricamento OFFLINE forzato da: /content/drive/MyDrive/INAIL_Thesis_Data_old/EmbeddingGemma_Offline
✓ Modello EmbeddingGemma locale caricato con successo (HPC ready)
  Device: cpu
[TF-IDF+EmbedRank] Inizializzazione per OSH (English)...
[TF-IDF+EmbedRank] Modello attivo: unknown
[TF-IDF+EmbedRank] ✓ Pronto per elaborazione inglese
✓ Ready

[Combo 1/6] Weights: [0.3, 0.3, 0.25, 0.15]
  → Avg confidence: 0.865 (valid: 50/50)

[Combo 2/6] Weights: [0.25, 0.35, 0.25, 0.15]
  → Avg confidence: 0.856 (valid: 50/50)

[Combo 3/6] Weights: [0.35, 0.25, 0.2, 0.2]
  → Avg confidence: 0.886 (valid: 50/50)

[Combo 4/6] Weights: [0.2, 0.4, 0.2, 0.2]
  → Avg confidence: 0.859 (valid: 50/50)

[Combo 5/6] Wei

In [ ]:
# Run
if __name__ == "__main__":
    run_weight_tests()


OSHA - WEIGHT OPTIMIZATION TEST

Options:
  [k] Keyword weights (50 docs, fast)
  [g] Global weights (30 docs, slower)
  [b] Both tests (sequential)
  [q] Quit

Choice (default q): g

TEST GLOBAL WEIGHTS - OSHA
Testing 30 documents (full enrichment)
⚠ NOTE: Adaptive scoring may override weights for low-law docs

[Setup] Initializing components...

[Embedding Model] Tentativo caricamento OFFLINE forzato da: /content/drive/MyDrive/INAIL_Thesis_Data_old/EmbeddingGemma_Offline
✓ Modello EmbeddingGemma locale caricato con successo (HPC ready)
  Device: cpu
[TF-IDF+EmbedRank] Inizializzazione per OSH (English)...
[TF-IDF+EmbedRank] Modello attivo: unknown
[TF-IDF+EmbedRank] ✓ Pronto per elaborazione inglese
[Professions] Pre-computing embeddings...
[Professions] ✓ Pronto
✓ Ready

[Combo 1/6] Weights: [0.45, 0.3, 0.25]
  → Avg confidence: 0.734 (valid: 30/30, adaptive: 0)

[Combo 2/6] Weights: [0.5, 0.25, 0.25]
  → Avg confidence: 0.742 (valid: 30/30, adaptive: 0)

[Combo 3/6] Weights: [0.35